In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt     
import seaborn as sns
sns.set_theme(style="whitegrid")

Ye sirf ek naya feautre adding cart first time ke liye h!!

In [2]:
final_df=pd.read_csv("Final_Cleaned_Dataset_with_sorting.csv")

In [3]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6727451 entries, 0 to 6727450
Data columns (total 12 columns):
 #   Column                Dtype  
---  ------                -----  
 0   event_time            str    
 1   event_type            str    
 2   product_id            int64  
 3   category_id           int64  
 4   category_code         str    
 5   brand                 str    
 6   price                 float64
 7   user_id               int64  
 8   user_session          str    
 9   category_code_filled  str    
 10  brand_filled          str    
 11  hour                  int64  
dtypes: float64(1), int64(4), str(7)
memory usage: 615.9 MB


In [4]:
final_df['event_time'] = pd.to_datetime(final_df['event_time'])

In [24]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6727451 entries, 0 to 6727450
Data columns (total 12 columns):
 #   Column                Dtype              
---  ------                -----              
 0   event_time            datetime64[us, UTC]
 1   event_type            str                
 2   product_id            int64              
 3   category_id           int64              
 4   category_code         str                
 5   brand                 str                
 6   price                 float64            
 7   user_id               int64              
 8   user_session          str                
 9   category_code_filled  str                
 10  brand_filled          str                
 11  hour                  int64              
dtypes: datetime64[us, UTC](1), float64(1), int64(4), str(6)
memory usage: 615.9 MB


In [49]:
print("🔍 VERIFYING THE 'MEDIAN CART PRICE' DECISION...")

# 1. Sirf un sessions ko nikalna jahan cart mein 1 se zyada items daale gaye
# (Kyunki 1 item hone par Mean aur Median same hi hota hai)
cart_events = final_df[final_df['event_type'] == 'cart']
cart_stats = cart_events.groupby('user_session').agg(
    cart_items_count=('price', 'count'),
    mean_cart_price=('price', 'mean'),
    median_cart_price=('price', 'median')
)

# Sirf multi-item carts par focus
multi_item_carts = cart_stats[cart_stats['cart_items_count'] > 1].copy()

# 2. Dhoka (Error) Calculate Karna
multi_item_carts['price_difference'] = abs(multi_item_carts['mean_cart_price'] - multi_item_carts['median_cart_price'])

# 3. Stats Print Karna
total_multi_carts = len(multi_item_carts)
big_difference = len(multi_item_carts[multi_item_carts['price_difference'] > 50]) # $50 se zyada ka farq

print("-" * 65)
print(f"Total Sessions with Multiple Items in Cart: {total_multi_carts:,}")
print(f"🚨 Carts where Mean was deceiving by > $50: {big_difference:,} ({big_difference/total_multi_carts*100:.2f}%)")
print("-" * 65)

# Extreme example dikhana jahan aapka decision model ko bacha lega
print("\n👀 EXTREME CASES (Why Median saved the model):")
display(multi_item_carts.sort_values(by='price_difference', ascending=False).head(3))

🔍 VERIFYING THE 'MEDIAN CART PRICE' DECISION...
-----------------------------------------------------------------
Total Sessions with Multiple Items in Cart: 60,284
🚨 Carts where Mean was deceiving by > $50: 2,569 (4.26%)
-----------------------------------------------------------------

👀 EXTREME CASES (Why Median saved the model):


,cart_items_count,mean_cart_price,median_cart_price,price_difference
user_session,,,,
fad4f9cb-1c34-4d0c-9f8b-25dd636bd751,3,1014.513333,256.89,757.623333
6ab9bc11-b5da-4f22-93ba-750cdab42aaf,7,872.287143,195.35,676.937143
a54aea6a-401d-4b1d-9af6-e0261f2240b9,7,740.151429,69.24,670.911429


In [50]:
import numpy as np
import pandas as pd

print("⚙️ Initiating Final Master Data Engineering Pipeline...")

# STEP 1: Memory Optimization
final_df['is_view'] = (final_df['event_type'] == 'view').astype(np.int8)
final_df['is_cart'] = (final_df['event_type'] == 'cart').astype(np.int8)
final_df['is_purchase'] = (final_df['event_type'] == 'purchase').astype(np.int8)

# Prices split (Prevents Data Leakage)
final_df['view_price'] = np.where(final_df['event_type'] == 'view', final_df['price'], np.nan)
final_df['cart_price'] = np.where(final_df['event_type'] == 'cart', final_df['price'], np.nan)

# Time to First Cart Prep
first_cart_time = final_df[final_df['is_cart'] == 1].groupby('user_session')['event_time'].min()

# STEP 2: The Master GroupBy (With Median Cart Price)
ml_dataframe = final_df.groupby('user_session').agg(
    total_views=('is_view', 'sum'),
    total_carts=('is_cart', 'sum'),
    unique_products=('product_id', 'nunique'),
    
    median_view_price=('view_price', 'median'),  
    median_cart_price=('cart_price', 'median'),  # 🔥 CHANGED: Average se Median kar diya
    
    start_time=('event_time', 'min'),
    end_time=('event_time', 'max'),
    target_purchase=('is_purchase', 'max')
)

# Missing values fill karna
ml_dataframe[['median_view_price', 'median_cart_price']] = ml_dataframe[['median_view_price', 'median_cart_price']].fillna(0)

# STEP 3: Duration & Velocity Features
ml_dataframe['session_duration_seconds'] = (ml_dataframe['end_time'] - ml_dataframe['start_time']).dt.total_seconds()
ml_dataframe['views_per_minute'] = ml_dataframe['total_views'] / ((ml_dataframe['session_duration_seconds'] + 1) / 60)

# --- THE NEW SUPER FEATURE: Focus Ratio ---
# Agar unique products 0 hain (jo practically impossible hai), toh division by zero se bachne ke liye thoda safe code
ml_dataframe['focus_ratio'] = ml_dataframe['total_views'] / ml_dataframe['unique_products'].replace(0, 1)

# Time to First Cart logic
ml_dataframe['first_cart_time_seconds'] = (first_cart_time - ml_dataframe['start_time']).dt.total_seconds()
ml_dataframe['first_cart_time_seconds'] = ml_dataframe['first_cart_time_seconds'].fillna(0)

# STEP 4: Cleaning Up
ml_dataframe = ml_dataframe.drop(columns=['start_time', 'end_time'])

print("✅ ml_dataframe is perfectly engineered with Focus Ratio & Median Cart Price!")
display(ml_dataframe.head())

⚙️ Initiating Final Master Data Engineering Pipeline...
✅ ml_dataframe is perfectly engineered with Focus Ratio & Median Cart Price!


,total_views,total_carts,unique_products,median_view_price,median_cart_price,target_purchase,session_duration_seconds,views_per_minute,focus_ratio,first_cart_time_seconds
user_session,,,,,,,,,,
00000510-834a-498a-9bed-561a90c5125e,1,0,1,69.17,0.0,0,0.0,60.000000,1.0,0.0
000009c4-a1dd-4764-87d9-24f3d7e43c4f,9,0,5,716.76,0.0,0,485.0,1.111111,1.8,0.0
0000189b-6d2d-45bc-a65c-3a9d94e330a4,1,0,1,10.18,0.0,0,0.0,60.000000,1.0,0.0
000029a6-1986-4a71-8139-53669c1adaba,6,0,3,362.76,0.0,0,133.0,2.686567,2.0,0.0
00003dca-5a5a-4af2-86ef-9851ceb02a2b,2,0,1,127.15,0.0,0,56.0,2.105263,2.0,0.0


In [ ]:
#ml_dataframe=pd.read_csv('ML_dataframe.csv')

In [42]:
ml_dataframe.head(5)

,total_views,total_carts,unique_products,avg_view_price,median_view_price,avg_cart_price,target_purchase,session_duration_seconds,views_per_minute,first_cart_time_seconds
user_session,,,,,,,,,,
00000510-834a-498a-9bed-561a90c5125e,1,0,1,69.170000,69.17,0.0,0,0.0,60.000000,0.0
000009c4-a1dd-4764-87d9-24f3d7e43c4f,9,0,5,822.974444,716.76,0.0,0,485.0,1.111111,0.0
0000189b-6d2d-45bc-a65c-3a9d94e330a4,1,0,1,10.180000,10.18,0.0,0,0.0,60.000000,0.0
000029a6-1986-4a71-8139-53669c1adaba,6,0,3,362.676667,362.76,0.0,0,133.0,2.686567,0.0
00003dca-5a5a-4af2-86ef-9851ceb02a2b,2,0,1,127.150000,127.15,0.0,0,56.0,2.105263,0.0


In [28]:
ml_dataframe.shape

(1378222, 11)

In [41]:
ml_dataframe=ml_dataframe.drop(columns=['start_time','end_time'])

KeyError: "['start_time', 'end_time'] not found in axis"

In [31]:
ml_dataframe[ml_dataframe['session_duration_seconds']==0]

,total_views,total_carts,unique_products,avg_view_price,avg_cart_price,target_purchase,session_duration_seconds,views_per_minute,first_cart_time_seconds
user_session,,,,,,,,,
00000510-834a-498a-9bed-561a90c5125e,1,0,1,69.17,0.0,0,0.0,60.0,0.0
0000189b-6d2d-45bc-a65c-3a9d94e330a4,1,0,1,10.18,0.0,0,0.0,60.0,0.0
000077fb-5f32-4a86-a179-2bbf8c34e9d8,1,0,1,159.59,0.0,0,0.0,60.0,0.0
0000ae8f-570d-4a89-9690-6f22050b8175,1,0,1,180.13,0.0,0,0.0,60.0,0.0
0000be71-fe05-4b44-af8e-e8de041095d2,1,0,1,372.73,0.0,0,0.0,60.0,0.0
...,...,...,...,...,...,...,...,...,...
ffff6769-207a-4cc1-a703-aba733aa0df1,1,0,1,574.02,0.0,0,0.0,60.0,0.0
ffffa9a9-336a-49f8-8ff3-8ebb37f00243,1,0,1,223.92,0.0,0,0.0,60.0,0.0
ffffcdf8-f820-4b22-acee-d8c425b624fc,1,0,1,244.54,0.0,0,0.0,60.0,0.0


In [10]:
Session_duration_0sec_and_purchased=(ml_dataframe[(ml_dataframe['session_duration_seconds']==0)& ml_dataframe['target_purchase']==1].shape[0]/ml_dataframe.shape[0])*100

In [11]:
Session_duration_0sec_and_purchased

0.023871335677416266

In [8]:
Session_duration_0sec=(ml_dataframe[ml_dataframe['session_duration_seconds']==0].shape[0]/ml_dataframe.shape[0])*100

In [9]:
Session_duration_0sec

36.363590190840085

In [12]:
ml_dataframe.head(4)

,Unnamed: 0,user_session,total_views,total_carts,unique_products,avg_view_price,avg_cart_price,target_purchase,session_duration_seconds
0,0,00000510-834a-498a-9bed-561a90c5125e,1,0,1,69.170000,0.0,0,0.0
1,1,000009c4-a1dd-4764-87d9-24f3d7e43c4f,9,0,5,822.974444,0.0,0,485.0
2,2,0000189b-6d2d-45bc-a65c-3a9d94e330a4,1,0,1,10.180000,0.0,0,0.0
3,3,000029a6-1986-4a71-8139-53669c1adaba,6,0,3,362.676667,0.0,0,133.0


In [4]:
# Session duration ki sachai dekhne ke liye
print(ml_dataframe['session_duration_seconds'].describe())

# Top 10 sabse lambe sessions dekhna (Ghanton mein convert karke)
(ml_dataframe['session_duration_seconds'] / 3600).nlargest(10)

count    1.378222e+06
mean     9.343076e+02
std      2.071115e+04
min      0.000000e+00
25%      0.000000e+00
50%      5.900000e+01
75%      2.840000e+02
max      2.444829e+06
Name: session_duration_seconds, dtype: float64


133538     679.119167
1197192    662.300556
1065900    658.316944
1034512    654.200556
630589     648.981944
37442      643.850000
1299701    632.386111
486710     629.397222
1053625    621.060278
891361     621.049167
Name: session_duration_seconds, dtype: float64

ZOMBIES:-

In [43]:
# Un logon ka data dekhte hain jo 1 ghante (3600 sec) se zyada ruke hain
zombies = ml_dataframe[ml_dataframe['session_duration_seconds'] > 3600]

print(f"Total Zombie Sessions: {len(zombies)}")
print("\nIn Zombies ne average kitne views aur carts kiye hain?")
print(zombies[['total_views', 'total_carts']].describe())

Total Zombie Sessions: 21579

In Zombies ne average kitne views aur carts kiye hain?
        total_views   total_carts
count  21579.000000  21579.000000
mean      21.582279      0.652996
std       28.706134      1.918654
min        1.000000      0.000000
25%        4.000000      0.000000
50%       11.000000      0.000000
75%       29.000000      1.000000
max      918.000000    100.000000


In [44]:
ml_dataframe.head()

,total_views,total_carts,unique_products,avg_view_price,median_view_price,avg_cart_price,target_purchase,session_duration_seconds,views_per_minute,first_cart_time_seconds
user_session,,,,,,,,,,
00000510-834a-498a-9bed-561a90c5125e,1,0,1,69.170000,69.17,0.0,0,0.0,60.000000,0.0
000009c4-a1dd-4764-87d9-24f3d7e43c4f,9,0,5,822.974444,716.76,0.0,0,485.0,1.111111,0.0
0000189b-6d2d-45bc-a65c-3a9d94e330a4,1,0,1,10.180000,10.18,0.0,0,0.0,60.000000,0.0
000029a6-1986-4a71-8139-53669c1adaba,6,0,3,362.676667,362.76,0.0,0,133.0,2.686567,0.0
00003dca-5a5a-4af2-86ef-9851ceb02a2b,2,0,1,127.150000,127.15,0.0,0,56.0,2.105263,0.0


# Removing the bots

In [45]:
import numpy as np

print(f"Original Data Size: {ml_dataframe.shape[0]}")

# MAGIC STEP: Action Velocity Calculate Karna (Views per minute)
# (Time mein 1 second add kar rahe hain taaki 'Divide by Zero' ka error na aaye jahan duration 0 hai)
duration_in_minutes = (ml_dataframe['session_duration_seconds'] + 1) / 60
views_per_minute = ml_dataframe['total_views'] / duration_in_minutes

# STEP 1: DROP THE BOTS (The Smarter Condition)
# Condition 1: Speed bohot zyada hai (jaise har minute mein 30+ views, yani har 2 second mein 1 naya page)
# Condition 2: Total views bhi thode zyada hone chahiye (taaki 0 second wale genuine 2-3 views drop na ho jayein)
bot_condition = (views_per_minute > 30) & (ml_dataframe['total_views'] > 15)

ml_dataframe = ml_dataframe[~bot_condition] # Bots ko uda do

print(f"Data Size after removing Fast-Clicking Bots: {ml_dataframe.shape[0]}")


# STEP 2: CAP THE ZOMBIE HUMANS (Baaki normal insaano ka time max 1 ghanta kar do)
ml_dataframe['session_duration_seconds'] = np.clip(
    ml_dataframe['session_duration_seconds'], 
    a_min=0, 
    a_max=3600
)

print(f"Maximum session duration ab hai: {ml_dataframe['session_duration_seconds'].max()} seconds")

Original Data Size: 1378222
Data Size after removing Fast-Clicking Bots: 1378222
Maximum session duration ab hai: 3600.0 seconds


CLIPPED_THE_TIME:-

In [46]:
ml_dataframe['session_duration_seconds'].describe()

count    1.378222e+06
mean     2.969126e+02
std      6.244187e+02
min      0.000000e+00
25%      0.000000e+00
50%      5.900000e+01
75%      2.840000e+02
max      3.600000e+03
Name: session_duration_seconds, dtype: float64

In [47]:
# Session duration ki sachai dekhne ke liye
print(ml_dataframe['session_duration_seconds'].describe())

# Top 10 sabse lambe sessions dekhna (Ghanton mein convert karke)
(ml_dataframe['session_duration_seconds'] / 3600).nlargest(10)

count    1.378222e+06
mean     2.969126e+02
std      6.244187e+02
min      0.000000e+00
25%      0.000000e+00
50%      5.900000e+01
75%      2.840000e+02
max      3.600000e+03
Name: session_duration_seconds, dtype: float64


user_session
0001b609-dbb0-4c1e-a036-2da4048f5760    1.0
000396ce-ac40-49b2-bce6-be288d13d251    1.0
0004947d-f145-468c-8f8d-84a5f1d76187    1.0
00074e23-b9c1-483b-8ffd-6c0914a10973    1.0
000917c6-5f55-41ce-a56f-d98750e4c484    1.0
000b52ba-d4d9-4d34-809d-95ba1a10e713    1.0
000d6780-3f8a-44e7-8089-1d29fa52b1f3    1.0
0015860d-7ca7-4b6e-b2fb-9ef3716e9249    1.0
00159c2c-ec6f-45d1-a7bd-7f813130fddc    1.0
00196ddc-f341-4a9f-996c-a2df162aded7    1.0
Name: session_duration_seconds, dtype: float64

In [48]:
ml_dataframe.to_csv("ml_dataframe_clipped_with_new_columns_median_view_Price.csv", index=False)    

In [53]:
ml_dataframe.to_csv("ml_dataframe_clipped_with_medain_view_cart_view_focus_ratio",index=False)

In [39]:
ml_dataframe

,total_views,total_carts,unique_products,avg_view_price,avg_cart_price,target_purchase,session_duration_seconds,views_per_minute,first_cart_time_seconds
user_session,,,,,,,,,
00000510-834a-498a-9bed-561a90c5125e,1,0,1,69.170000,0.00,0,0.0,60.000000,0.0
000009c4-a1dd-4764-87d9-24f3d7e43c4f,9,0,5,822.974444,0.00,0,485.0,1.111111,0.0
0000189b-6d2d-45bc-a65c-3a9d94e330a4,1,0,1,10.180000,0.00,0,0.0,60.000000,0.0
000029a6-1986-4a71-8139-53669c1adaba,6,0,3,362.676667,0.00,0,133.0,2.686567,0.0
00003dca-5a5a-4af2-86ef-9851ceb02a2b,2,0,1,127.150000,0.00,0,56.0,2.105263,0.0
...,...,...,...,...,...,...,...,...,...
ffffb94a-cea4-429d-ba4a-7e47a6f28171,12,1,9,809.506667,977.86,1,801.0,0.897756,721.0
ffffb9ea-87d1-4f3d-b107-91b1bb087d36,4,1,4,59.842500,32.15,0,177.0,1.348315,177.0
ffffcdf8-f820-4b22-acee-d8c425b624fc,1,0,1,244.540000,0.00,0,0.0,60.000000,0.0


# End Code for Pipelining:-

In [51]:
import numpy as np
import pandas as pd

print("⚙️ Initiating Ultimate Master Data Pipeline (With Anti-Bot & Capping)...")

# ==========================================
# STEP 1: PREPARATION & MEMORY OPTIMIZATION
# ==========================================
final_df['is_view'] = (final_df['event_type'] == 'view').astype(np.int8)
final_df['is_cart'] = (final_df['event_type'] == 'cart').astype(np.int8)
final_df['is_purchase'] = (final_df['event_type'] == 'purchase').astype(np.int8)

# Prices split (Prevents Data Leakage)
final_df['view_price'] = np.where(final_df['event_type'] == 'view', final_df['price'], np.nan)
final_df['cart_price'] = np.where(final_df['event_type'] == 'cart', final_df['price'], np.nan)

# Time to First Cart Prep
first_cart_time = final_df[final_df['is_cart'] == 1].groupby('user_session')['event_time'].min()

# ==========================================
# STEP 2: THE MASTER GROUPBY
# ==========================================
ml_dataframe = final_df.groupby('user_session').agg(
    total_views=('is_view', 'sum'),
    total_carts=('is_cart', 'sum'),
    unique_products=('product_id', 'nunique'),
    
    median_view_price=('view_price', 'median'),  
    median_cart_price=('cart_price', 'median'),  
    
    start_time=('event_time', 'min'),
    end_time=('event_time', 'max'),
    target_purchase=('is_purchase', 'max')
)

# Missing values fill karna
ml_dataframe[['median_view_price', 'median_cart_price']] = ml_dataframe[['median_view_price', 'median_cart_price']].fillna(0)

# Raw Duration Calculate karna (Abhi cap nahi kiya)
ml_dataframe['session_duration_seconds'] = (ml_dataframe['end_time'] - ml_dataframe['start_time']).dt.total_seconds()

# ==========================================
# STEP 3: DROP THE BOTS (Using Raw Data)
# ==========================================
print(f"Original Data Size: {ml_dataframe.shape[0]:,}")

# Temporary raw speed for checking bots
raw_duration_minutes = (ml_dataframe['session_duration_seconds'] + 1) / 60
raw_views_per_minute = ml_dataframe['total_views'] / raw_duration_minutes

# Drop condition (Fast-Clicking Bots)
bot_condition = (raw_views_per_minute > 30) & (ml_dataframe['total_views'] > 15)
ml_dataframe = ml_dataframe[~bot_condition] 

print(f"Data Size after removing Bots: {ml_dataframe.shape[0]:,}")

# ==========================================
# STEP 4: CAP THE ZOMBIE HUMANS
# ==========================================
ml_dataframe['session_duration_seconds'] = np.clip(
    ml_dataframe['session_duration_seconds'], 
    a_min=0, 
    a_max=3600
)
print(f"Maximum session duration capped to: {ml_dataframe['session_duration_seconds'].max()} seconds")

# ==========================================
# STEP 5: FINAL SUPER FEATURES (Using Clean Data)
# ==========================================
# Ab hum final velocity nikalenge capped time ke sath (Ab error nahi aayega!)
ml_dataframe['views_per_minute'] = ml_dataframe['total_views'] / ((ml_dataframe['session_duration_seconds'] + 1) / 60)

# Focus Ratio
ml_dataframe['focus_ratio'] = ml_dataframe['total_views'] / ml_dataframe['unique_products'].replace(0, 1)

# Time to First Cart
ml_dataframe['first_cart_time_seconds'] = (first_cart_time - ml_dataframe['start_time']).dt.total_seconds()
ml_dataframe['first_cart_time_seconds'] = ml_dataframe['first_cart_time_seconds'].fillna(0)

# Cleanup
ml_dataframe = ml_dataframe.drop(columns=['start_time', 'end_time'])

print("\n✅ Final ml_dataframe is perfectly engineered, cleaned, and ready for ML!")
display(ml_dataframe.head())

⚙️ Initiating Ultimate Master Data Pipeline (With Anti-Bot & Capping)...
Original Data Size: 1,378,222
Data Size after removing Bots: 1,378,222
Maximum session duration capped to: 3600.0 seconds

✅ Final ml_dataframe is perfectly engineered, cleaned, and ready for ML!


,total_views,total_carts,unique_products,median_view_price,median_cart_price,target_purchase,session_duration_seconds,views_per_minute,focus_ratio,first_cart_time_seconds
user_session,,,,,,,,,,
00000510-834a-498a-9bed-561a90c5125e,1,0,1,69.17,0.0,0,0.0,60.000000,1.0,0.0
000009c4-a1dd-4764-87d9-24f3d7e43c4f,9,0,5,716.76,0.0,0,485.0,1.111111,1.8,0.0
0000189b-6d2d-45bc-a65c-3a9d94e330a4,1,0,1,10.18,0.0,0,0.0,60.000000,1.0,0.0
000029a6-1986-4a71-8139-53669c1adaba,6,0,3,362.76,0.0,0,133.0,2.686567,2.0,0.0
00003dca-5a5a-4af2-86ef-9851ceb02a2b,2,0,1,127.15,0.0,0,56.0,2.105263,2.0,0.0


In [8]:
ml_dataframe.to_csv('ml_dataframe_clipped_with_medain_view_cart_view_focus_ratio_other_features.csv')

In [6]:
import numpy as np
import pandas as pd
from IPython.display import display

print("⚙️ Initiating Ultimate Master Data Pipeline (With Anti-Bot, Capping & Super Features)...")

# ==========================================
# STEP 1: PREPARATION & MEMORY OPTIMIZATION
# ==========================================
# Encoding event types for faster processing
final_df['is_view'] = (final_df['event_type'] == 'view').astype(np.int8)
final_df['is_cart'] = (final_df['event_type'] == 'cart').astype(np.int8)
final_df['is_purchase'] = (final_df['event_type'] == 'purchase').astype(np.int8)

# Prices split (Prevents Data Leakage & helps in Budget Clarity)
final_df['view_price'] = np.where(final_df['event_type'] == 'view', final_df['price'], np.nan)
final_df['cart_price'] = np.where(final_df['event_type'] == 'cart', final_df['price'], np.nan)

# Time to First Cart Prep
first_cart_time = final_df[final_df['is_cart'] == 1].groupby('user_session')['event_time'].min()

# ==========================================
# STEP 2: THE MASTER GROUPBY
# ==========================================
ml_dataframe = final_df.groupby('user_session').agg(
    total_views=('is_view', 'sum'),
    total_carts=('is_cart', 'sum'),
    unique_products=('product_id', 'nunique'),
    
    median_view_price=('view_price', 'median'),  
    median_cart_price=('cart_price', 'median'), 
    
    # Min and Max view prices for Budget Clarity
    max_view_price=('view_price', 'max'),
    min_view_price=('view_price', 'min'),
    
    start_time=('event_time', 'min'),
    end_time=('event_time', 'max'),
    target_purchase=('is_purchase', 'max')
)

# Fill missing values
ml_dataframe[['median_view_price', 'median_cart_price']] = ml_dataframe[['median_view_price', 'median_cart_price']].fillna(0)
ml_dataframe[['max_view_price', 'min_view_price']] = ml_dataframe[['max_view_price', 'min_view_price']].fillna(0)

# Raw Duration Calculation
ml_dataframe['session_duration_seconds'] = (ml_dataframe['end_time'] - ml_dataframe['start_time']).dt.total_seconds()

# ==========================================
# STEP 3: DROP THE BOTS (Using Raw Data)
# ==========================================
print(f"Original Data Size: {ml_dataframe.shape[0]:,}")

# Temporary raw speed for checking bots
raw_duration_minutes = (ml_dataframe['session_duration_seconds'] + 1) / 60
raw_views_per_minute = ml_dataframe['total_views'] / raw_duration_minutes

# Drop condition (Fast-Clicking Bots)
bot_condition = (raw_views_per_minute > 30) & (ml_dataframe['total_views'] > 15)
ml_dataframe = ml_dataframe[~bot_condition] 

print(f"Data Size after removing Bots: {ml_dataframe.shape[0]:,}")

# ==========================================
# STEP 4: CAP THE ZOMBIE HUMANS
# ==========================================
ml_dataframe['session_duration_seconds'] = np.clip(
    ml_dataframe['session_duration_seconds'], 
    a_min=0, 
    a_max=3600 # 1 Hour cap
)
print(f"Maximum session duration capped to: {ml_dataframe['session_duration_seconds'].max()} seconds")

# ==========================================
# STEP 5: FINAL SUPER FEATURES 
# ==========================================
# 1. Base Features
ml_dataframe['views_per_minute'] = ml_dataframe['total_views'] / ((ml_dataframe['session_duration_seconds'] + 1) / 60)
ml_dataframe['focus_ratio'] = ml_dataframe['total_views'] / ml_dataframe['unique_products'].replace(0, 1)

# 2. FEATURE I: Time to First Cart
ml_dataframe['first_cart_time_seconds'] = (first_cart_time - ml_dataframe['start_time']).dt.total_seconds()
ml_dataframe['first_cart_time_seconds'] = ml_dataframe['first_cart_time_seconds'].fillna(0)
ml_dataframe['first_cart_time_seconds'] = np.clip(ml_dataframe['first_cart_time_seconds'], a_min=0, a_max=3600)

# 3. FEATURE II: Post-Cart Hesitation (Dwell Time)
ml_dataframe['post_cart_dwell_seconds'] = np.where(
    ml_dataframe['total_carts'] > 0,
    ml_dataframe['session_duration_seconds'] - ml_dataframe['first_cart_time_seconds'],
    0
)
ml_dataframe['post_cart_dwell_seconds'] = np.clip(ml_dataframe['post_cart_dwell_seconds'], a_min=0, a_max=3600)

# 4. FEATURE III: Budget Clarity (Price Exploration Range)
ml_dataframe['price_exploration_range'] = ml_dataframe['max_view_price'] - ml_dataframe['min_view_price']
CAP_PRICE = 1000
ml_dataframe['price_exploration_capped'] = np.clip(ml_dataframe['price_exploration_range'], a_min=0, a_max=CAP_PRICE)

# 5. FEATURE IV: Decisiveness Metric (Cart to View Ratio)
# Formula: Total_Carts / Total_Views
# Replacing 0 views with 1 to avoid ZeroDivisionError
ml_dataframe['cart_view_ratio'] = ml_dataframe['total_carts'] / ml_dataframe['total_views'].replace(0, 1)
# Capping at 1.0 just in case there are rare sessions with more carts than views
ml_dataframe['cart_view_ratio'] = np.clip(ml_dataframe['cart_view_ratio'], a_min=0, a_max=1.0)

# ==========================================
# STEP 6: CLEANUP
# ==========================================
# Drop intermediate columns
columns_to_drop = ['start_time', 'end_time', 'max_view_price', 'min_view_price', 'price_exploration_range']
ml_dataframe = ml_dataframe.drop(columns=columns_to_drop)

print("\n✅ Final ml_dataframe is perfectly engineered, cleaned, and ready for ML!")
display(ml_dataframe.head())

⚙️ Initiating Ultimate Master Data Pipeline (With Anti-Bot, Capping & Super Features)...
Original Data Size: 1,378,222
Data Size after removing Bots: 1,378,222
Maximum session duration capped to: 3600.0 seconds

✅ Final ml_dataframe is perfectly engineered, cleaned, and ready for ML!


,total_views,total_carts,unique_products,median_view_price,median_cart_price,target_purchase,session_duration_seconds,views_per_minute,focus_ratio,first_cart_time_seconds,post_cart_dwell_seconds,price_exploration_capped,cart_view_ratio
user_session,,,,,,,,,,,,,
00000510-834a-498a-9bed-561a90c5125e,1,0,1,69.17,0.0,0,0.0,60.000000,1.0,0.0,0.0,0.00,0.0
000009c4-a1dd-4764-87d9-24f3d7e43c4f,9,0,5,716.76,0.0,0,485.0,1.111111,1.8,0.0,0.0,871.19,0.0
0000189b-6d2d-45bc-a65c-3a9d94e330a4,1,0,1,10.18,0.0,0,0.0,60.000000,1.0,0.0,0.0,0.00,0.0
000029a6-1986-4a71-8139-53669c1adaba,6,0,3,362.76,0.0,0,133.0,2.686567,2.0,0.0,0.0,130.29,0.0
00003dca-5a5a-4af2-86ef-9851ceb02a2b,2,0,1,127.15,0.0,0,56.0,2.105263,2.0,0.0,0.0,0.00,0.0
